# 🚀 Phase 3 : Modèle Hybride Ultime (Collaboratif + Content + Popularité)

## 🎯 Objectif du Notebook
L'objectif de cette dernière étape de modélisation est de combiner les forces de nos différents algorithmes pour créer un moteur de recommandation robuste, polyvalent et sans faille.

Nous allons fusionner trois signaux complémentaires :
1. **L'ALS (Filtrage Collaboratif) [Hit Ratio: 14.39%]** : Capte les schémas de navigation complexes des utilisateurs.
2. **Le Content-Based (Similarité Cosinus) [Hit Ratio: 10.30%]** : Analyse le sujet sémantique. Résout le problème du **Cold-Start Article** (recommander un article nouveau qui n'a pas encore de clics).
3. **Le Popularity Baseline (Time Decay)** : Met en avant les tendances récentes. Résout le problème du **Cold-Start Utilisateur** (un nouvel utilisateur sans historique recevra les tendances du moment).

## 🧠 Stratégie d'Hybridation (Pondération)
Puisque les scores de nos modèles n'ont pas la même échelle (produit scalaire arbitraire pour ALS, de 0 à 1 pour Cosinus, etc.), nous allons d'abord les **normaliser (MinMax Scaler)** entre 0 et 1.

Nous appliquerons ensuite une **Moyenne Pondérée (Weighted Sum)** pour obtenir le score final :

$$ Score\_Hybride = (\alpha \times Score\_ALS) + (\beta \times Score\_Content) + (\gamma \times Score\_Pop) $$

*(Avec $\alpha + \beta + \gamma = 1$)*

Cette formule offre une flexibilité totale à notre API : nous pourrons ajuster les poids selon nos objectifs (ex: augmenter $\gamma$ le matin si on veut pousser les actus "Breaking News").

Viewed Taches.txt:1-16
Edited Taches.txt

En regardant les mises à jour que tu viens de faire dans ton fichier `Taches.txt`, je comprends exactement ce que tu voulais dire par "étudier les cas des users" ! 

Tu as écrit : *"Afficher un graphe avec les scores de hit ratio pour les nouveaux utilisateurs et anciens utilisateurs"* et *"Séparer les cas des users (regrouper les users <4 clicks dans un cas , ….)"*.

C'est une démarche d'analyse de la donnée (Data Analysis) absolument parfaite. Au lieu de se contenter d'un Hit Ratio global, on va regarder **qui profite de quel algorithme**.

Voici donc les 4 grandes étapes que nous allons coder dans ce **Notebook 3** :

### 🛠️ Étape 1 : Le rassemblement des Cerveaux
* On charge nos 3 modèles entraînés : la matrice de l'ALS, les Embeddings PCA du Content-Based, et le dataframe de Popularité.
* On charge l'historique des clics.

### ⚙️ Étape 2 : Création de la Fonction Hybride
* On crée la fameuse formule : `(𝛼 × ALS) + (𝛽 × Content) + (𝛾 × Popularité)`.
* On utilise un *MinMax Scaler* pour que tout le monde parle le même langage (entre 0 et 1).

### 📊 Étape 3 : L'Évaluation Quantitative par Segments (Ta tâche principale !)
* On sépare nos 10 000 utilisateurs de test en deux groupes :
  1. **Les "Cold" (Nouveaux) :** Ceux qui ont moins de 5 clics d'historique.
  2. **Les "Warm" (Anciens) :** Ceux qui ont 5 clics ou plus.
* On calcule le Hit Ratio de chaque modèle isolé + du modèle Hybride sur ces deux groupes.
* On trace **LE graphique final** (Bar Chart). C'est là qu'on verra l'ALS s'effondrer sur les nouveaux utilisateurs, et l'Hybride triompher sur tous les tableaux.

### 🧐 Étape 4 : L'Évaluation Qualitative (Cas d'usages concrets)
* Pour illustrer le graphique, on prend "à la main" un utilisateur "Cold" et un utilisateur "Warm".
* On affiche la catégorie de leurs lectures passées, et on affiche ce que le modèle hybride a décidé de leur recommander pour voir si c'est pertinent.

C'est un plan en béton armé. Si tu es d'accord avec ça, on attaque la **Cellule de l'Étape 1** !

Sommaire:

- Fonction de recommandation de chaque approche prédefinit
    - ALS
    - Cosinus
    - Popularité

- Fonction de recommandation hybdride

- Évaluation quantitative générale (10000 utilisateurs)

- Évaluation quantitative par segments (cold(< 5 clicks) vs warm(> 5 clicks))

- Évaluation qualitative (cas d'usages concrets)

# Start

In [7]:
# ==========================================
# 1. IMPORTATION ET PRÉPARATION DES DONNÉES
# ==========================================
!pip install implicit pyarrow scikit-learn

import pandas as pd
import numpy as np
import implicit
import pickle
import time
import os
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

from google.colab import drive
drive.mount('/content/drive')

# Chemins
DATA_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Data"
GENERATED_DIR = "/content/drive/MyDrive/OPENCLASSROOMS/Projets/P10/Generated"

print("📂 Chargement des cerveaux de nos modèles...")

# 1. Clics & Metadata
clicks_df = pd.read_parquet(os.path.join(GENERATED_DIR, "clicks_full.parquet"))
articles_df = pd.read_csv(os.path.join(DATA_DIR, "articles_metadata.csv"))

# 2. Modèle ALS (Collaboratif)
user_factors = np.load(os.path.join(GENERATED_DIR, "Collaborative_best", "als_user_factors.npy"))
item_factors = np.load(os.path.join(GENERATED_DIR, "Collaborative_best", "als_item_factors.npy"))

# Reconstruction du modèle Implicit
model_als = implicit.als.AlternatingLeastSquares(factors=50)
model_als.user_factors = user_factors
model_als.item_factors = item_factors

# Chargement des Mappings (Tableaux Numpy)
with open(os.path.join(GENERATED_DIR, "Collaborative_best", "als_user_mapping.pkl"), "rb") as f:
    user_array = pickle.load(f)
with open(os.path.join(GENERATED_DIR, "Collaborative_best", "als_item_mapping.pkl"), "rb") as f:
    item_array = pickle.load(f)
    
# 1. Traducteur Utilisateur (Vrai ID -> ID Interne ALS)
real_to_internal_user = {real_id: idx for idx, real_id in enumerate(user_array)}

# 2. Traducteur Article (Vrai ID -> ID Interne ALS)
real_to_internal_item = {real_id: idx for idx, real_id in enumerate(item_array)}

# 3. Traducteur Article Inverse (ID Interne ALS -> Vrai ID)
# Puisque item_array est déjà un tableau, on fera simplement : vrai_id = item_array[id_interne]
internal_to_real_item = item_array

# 3. Embeddings PCA (Content-Based)
with open(os.path.join(GENERATED_DIR, "articles_embeddings_pca.pickle"), "rb") as f:
    embeddings_pca = pickle.load(f)

# 4. Modèle Popularité (Time Decay)
popularity_df = pd.read_parquet(os.path.join(GENERATED_DIR, "Popularity_data", "articles_popularity_time_decay.parquet"))
dict_popularity = dict(zip(popularity_df['click_article_id'], popularity_df['time_decay_score']))

print("✅ Tous les modèles (et leurs mappings) sont chargés et prêts pour l'Hybridation !")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Chargement des cerveaux de nos modèles...
✅ Tous les modèles (et leurs mappings) sont chargés et prêts pour l'Hybridation !


# Fonctions de recommandation

In [18]:
# ==========================================
# 2. LES 3 MOTEURS DE RECOMMANDATION ISOLÉS
# ==========================================

print("🎯 Création des fonctions de recommandation individuelles...")

import scipy.sparse as sparse
print("📂 Chargement de la matrice creuse (CSR)...")
# On charge la matrice Item-User et on la transpose en User-Item (Format attendu par implicit)
sparse_item_user = sparse.load_npz(os.path.join(GENERATED_DIR, "sparse_item_user_matrix.npz"))
user_item_matrix = sparse_item_user.T.tocsr()
print("✅ Matrice chargée !")

import scipy.sparse as sparse

# --- MOTEUR 1 : FILTRAGE COLLABORATIF (ALS) ---
def get_als_recommendations(user_id, top_n=50):
    """ Renvoie les recommandations personnalisées selon le comportement. """
    if user_id not in real_to_internal_user:
        return {} 
    
    internal_user_id = real_to_internal_user[user_id]
    
    # 💡 L'ASTUCE : On crée une matrice vide à la volée, à la bonne taille exacte (41002)
    num_items = len(item_array)
    empty_matrix = sparse.csr_matrix((1, num_items))
    
    # On demande les recommandations SANS filtrer l'historique
    internal_item_ids, scores = model_als.recommend(
        internal_user_id, 
        empty_matrix, 
        N=top_n,
        filter_already_liked_items=False  # <-- On désactive le filtre !
    )
    
    recs = {}
    for int_id, score in zip(internal_item_ids, scores):
        real_article_id = internal_to_real_item[int_id]
        recs[real_article_id] = float(score)
        
    return recs




# --- MOTEUR 2 : CONTENT-BASED (COSINUS) ---
def get_content_based_recommendations(user_id, top_n=50):
    """ Renvoie des articles similaires sémantiquement au dernier article lu. """
    # 1. Trouver le dernier article lu par l'utilisateur
    user_history = clicks_df[clicks_df['user_id'] == user_id]
    if user_history.empty:
        return {} # L'utilisateur n'a jamais rien lu (Cold-Start)
    
    last_article_id = user_history.iloc[-1]['click_article_id']
    
    # 2. Extraire son ADN mathématique
    target_vector = embeddings_pca[last_article_id].reshape(1, -1)
    
    # 3. Calcul de toutes les similarités
    similarities = cosine_similarity(target_vector, embeddings_pca)[0]
    
    # 4. Trier pour prendre le Top 50 (en ignorant le 1er qui est l'article lui-même)
    top_indices = np.argsort(similarities)[::-1][1:top_n+1]
    scores = similarities[top_indices]
    
    # 5. Formatage en dictionnaire
    recs = {}
    for idx, score in zip(top_indices, scores):
        recs[idx] = float(score)
        
    return recs


# --- MOTEUR 3 : POPULARITÉ (TIME DECAY) ---
def get_popularity_recommendations(top_n=50):
    """ Renvoie les articles Tendance du moment (Joker Cold-Start). """
    # Trier le dictionnaire de popularité par ordre décroissant
    sorted_pop = sorted(dict_popularity.items(), key=lambda item: item[1], reverse=True)
    
    # Prendre les N premiers et convertir en dictionnaire
    return dict(sorted_pop[:top_n])


print("✅ Les 3 moteurs sont créés !")


🎯 Création des fonctions de recommandation individuelles...
📂 Chargement de la matrice creuse (CSR)...
✅ Matrice chargée !
✅ Les 3 moteurs sont créés !


In [19]:
# ==========================================
# 3. FONCTION DE RECOMMANDATION HYBRIDE
# ==========================================

print("🧬 Création du Moteur Hybride...")

def recommend_hybrid(user_id, top_n=5, weight_als=0.60, weight_cb=0.30, weight_pop=0.10):
    """
    Combine ALS, Content-Based et Popularité avec pondération.
    Normalise les scores pour qu'ils aient tous le même impact mathématique.
    """
    # 1. Récupération des recommandations brutes
    als_recs = get_als_recommendations(user_id, top_n=50)
    cb_recs = get_content_based_recommendations(user_id, top_n=50)
    pop_recs = get_popularity_recommendations(top_n=50)
    
    # 🛡️ Sécurité Cold-Start Total : 
    # Si l'utilisateur n'a AUCUN clic, on n'a ni ALS, ni Content. On renvoie les Tendances !
    if not als_recs and not cb_recs:
        return list(pop_recs.keys())[:top_n]
    
    # 2. On rassemble tous les articles proposés dans une même liste (sans doublons)
    all_articles = set(list(als_recs.keys()) + list(cb_recs.keys()) + list(pop_recs.keys()))
    
    # 3. Création du tableau de confrontation
    df_recs = pd.DataFrame(index=list(all_articles))
    df_recs['score_als'] = pd.Series(als_recs)
    df_recs['score_cb'] = pd.Series(cb_recs)
    df_recs['score_pop'] = pd.Series(pop_recs)
    
    # Si un article n'a pas été proposé par un modèle, il a 0
    df_recs = df_recs.fillna(0)
    
    # 4. Normalisation des scores (MinMax Scaler : de 0.0 à 1.0)
    scaler = MinMaxScaler()
    
    if df_recs['score_als'].max() > 0:
        df_recs['score_als'] = scaler.fit_transform(df_recs[['score_als']])
        
    if df_recs['score_cb'].max() > 0:
        df_recs['score_cb'] = scaler.fit_transform(df_recs[['score_cb']])
        
    if df_recs['score_pop'].max() > 0:
        df_recs['score_pop'] = scaler.fit_transform(df_recs[['score_pop']])
        
    # 5. Application de la Formule Hybride
    
    # 🛡️ Sécurité Cold-Start Partiel : 
    # Si l'ALS ne connait pas le User, on transfère ses 60% d'importance au Content-Based
    if not als_recs:
        w_cb = weight_cb + weight_als
        w_als = 0.0
    else:
        w_cb = weight_cb
        w_als = weight_als
        
    df_recs['score_hybrid'] = (df_recs['score_als'] * w_als) + \
                              (df_recs['score_cb'] * w_cb) + \
                              (df_recs['score_pop'] * weight_pop)
                              
    # 6. Tri décroissant sur le score hybride final
    df_recs = df_recs.sort_values('score_hybrid', ascending=False)
    
    # On renvoie uniquement les IDs du Top N
    return df_recs.head(top_n).index.tolist()

print("✅ Moteur Hybride Opérationnel !")


🧬 Création du Moteur Hybride...
✅ Moteur Hybride Opérationnel !


# Évaluation quantitative générale

In [20]:
# ==========================================
# 4. ÉVALUATION QUANTITATIVE (HYBRIDE & SEGMENTS)
# ==========================================
print("🛠️ Préparation des données d'évaluation...")

# 1. Préparation du Train/Test Split (Leave-One-Out)
user_counts = clicks_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index

df_valid = clicks_df[clicks_df['user_id'].isin(valid_users)].copy()
df_valid = df_valid.sort_values(['user_id', 'click_timestamp'])

test_set = df_valid.groupby('user_id').tail(1)

# Échantillon de 10 000 utilisateurs (Fixé avec random_state=42 pour comparer équitablement)
sample_users = test_set['user_id'].sample(10000, random_state=42).values

# 2. Définition de nos Segments Utilisateurs 
# (Cold = Moins de 5 clics dans tout leur historique | Warm = 5 clics ou plus)
cold_users_set = set(user_counts[user_counts < 5].index)

# Compteurs
hits_hybrid_global = 0
hits_hybrid_cold = 0
hits_hybrid_warm = 0

count_cold = 0
count_warm = 0

print("⏳ Lancement de l'évaluation sur 10 000 utilisateurs (Durée estimée : ~45 minutes)...")
t0 = time.time()

for user in sample_users:
    # L'article cible (celui qu'il a lu et qu'on a caché)
    target_article = test_set[test_set['user_id'] == user]['click_article_id'].values[0]
    
    # 💡 La Prédiction Hybride Ultime
    recs_hybrid = recommend_hybrid(user, top_n=5, weight_als=0.60, weight_cb=0.30, weight_pop=0.10)
    
    # Est-ce qu'on a juste ?
    is_hit = target_article in recs_hybrid
    
    # --- RANGEMENT DANS LES TIROIRS ---
    # 1. Global
    if is_hit:
        hits_hybrid_global += 1
        
    # 2. Segments
    if user in cold_users_set:
        count_cold += 1
        if is_hit:
            hits_hybrid_cold += 1
    else:
        count_warm += 1
        if is_hit:
            hits_hybrid_warm += 1

# ==========================================
# AFFICHAGE DES RÉSULTATS
# ==========================================
print("\n" + "="*60)
print("🏁 RÉSULTATS FINAUX DU MODÈLE HYBRIDE")
print("="*60)
print(f"🎯 Hit Ratio GLOBAL    : {(hits_hybrid_global / 10000) * 100:.2f} %")
print("-" * 60)
print(f"🧊 Hit Ratio COLD-START (< 5 clics)  | Sur {count_cold:<4} users | Score : {(hits_hybrid_cold / max(1, count_cold)) * 100:.2f} %")
print(f"🔥 Hit Ratio WARM       (>= 5 clics) | Sur {count_warm:<4} users | Score : {(hits_hybrid_warm / max(1, count_warm)) * 100:.2f} %")
print("-" * 60)
print(f"⏱️ Temps total d'évaluation : {time.time() - t0:.2f} s")


🛠️ Préparation des données d'évaluation...
⏳ Lancement de l'évaluation sur 10 000 utilisateurs (Durée estimée : ~45 minutes)...

🏁 RÉSULTATS FINAUX DU MODÈLE HYBRIDE
🎯 Hit Ratio GLOBAL    : 5.38 %
------------------------------------------------------------
🧊 Hit Ratio COLD-START (< 5 clics)  | Sur 5040 users | Score : 9.13 %
🔥 Hit Ratio WARM       (>= 5 clics) | Sur 4960 users | Score : 1.57 %
------------------------------------------------------------
⏱️ Temps total d'évaluation : 1963.01 s


# Évaluation quantitative par segments
(sur l'hybride c'est déja fait, faut le refaire pour les 2 autres techniques)

In [ ]:
# ==========================================
# 5. ÉVALUATION SEGMENTÉE (ALS & CONTENT-BASED)
# ==========================================
print("🛠️ Lancement de l'évaluation sur 10 000 utilisateurs pour ALS et Content-Based...")
t0 = time.time()

hits_als_global = 0
hits_als_cold = 0
hits_als_warm = 0

hits_cb_global = 0
hits_cb_cold = 0
hits_cb_warm = 0

count_cold = 0
count_warm = 0

for user in sample_users:
    target_article = test_set[test_set['user_id'] == user]['click_article_id'].values[0]
    
    # 💡 CORRECTION DU BUG ICI :
    # On prend tout l'historique de l'utilisateur...
    user_history = clicks_df[clicks_df['user_id'] == user]['click_article_id'].tolist()
    # ...Mais on n'interdit PAS la bonne réponse (puisque c'est celle qu'on veut qu'il trouve !)
    if target_article in user_history:
        user_history.remove(target_article)
        
    # On transforme en Set pour que la recherche soit 1000x plus rapide en Python
    forbidden_articles = set(user_history)
    
    # -------------------------
    # 1. PRÉDICTION ALS
    # -------------------------
    als_raw = get_als_recommendations(user, top_n=50)
    als_top5 = [art for art in als_raw.keys() if art not in forbidden_articles][:5]
    is_hit_als = target_article in als_top5
    
    # -------------------------
    # 2. PRÉDICTION CONTENT-BASED
    # -------------------------
    cb_raw = get_content_based_recommendations(user, top_n=50)
    cb_top5 = [art for art in cb_raw.keys() if art not in forbidden_articles][:5]
    is_hit_cb = target_article in cb_top5
    
    # -------------------------
    # RANGEMENT DES RÉSULTATS
    # -------------------------
    if is_hit_als: hits_als_global += 1
    if is_hit_cb: hits_cb_global += 1
        
    if user in cold_users_set:
        count_cold += 1
        if is_hit_als: hits_als_cold += 1
        if is_hit_cb: hits_cb_cold += 1
    else:
        count_warm += 1
        if is_hit_als: hits_als_warm += 1
        if is_hit_cb: hits_cb_warm += 1

# ==========================================
# AFFICHAGE DES RÉSULTATS COMPARATIFS
# ==========================================
print("\n" + "="*60)
print("🏁 RÉSULTATS DE L'ÉVALUATION PAR SEGMENTS")
print("="*60)

print(f"\n🥇 MOTEUR ALS (Filtrage Collaboratif)")
print(f"🎯 Hit Ratio GLOBAL : {(hits_als_global / 10000) * 100:.2f} %")
print(f"🧊 COLD-START (<5 clics) : {(hits_als_cold / max(1, count_cold)) * 100:.2f} %")
print(f"🔥 WARM (>=5 clics)      : {(hits_als_warm / max(1, count_warm)) * 100:.2f} %")

print(f"\n🥈 MOTEUR CONTENT-BASED (Cosinus)")
print(f"🎯 Hit Ratio GLOBAL : {(hits_cb_global / 10000) * 100:.2f} %")
print(f"🧊 COLD-START (<5 clics) : {(hits_cb_cold / max(1, count_cold)) * 100:.2f} %")
print(f"🔥 WARM (>=5 clics)      : {(hits_cb_warm / max(1, count_warm)) * 100:.2f} %")
print("-" * 60)
print(f"⏱️ Temps total d'évaluation : {time.time() - t0:.2f} s")


🛠️ Lancement de l'évaluation sur 10 000 utilisateurs pour ALS et Content-Based...

🏁 RÉSULTATS DE L'ÉVALUATION PAR SEGMENTS

🥇 MOTEUR ALS (Filtrage Collaboratif)
🎯 Hit Ratio GLOBAL : 0.00 %
🧊 COLD-START (<5 clics) : 0.00 %
🔥 WARM (>=5 clics)      : 0.00 %

🥈 MOTEUR CONTENT-BASED (Cosinus)
🎯 Hit Ratio GLOBAL : 0.00 %
🧊 COLD-START (<5 clics) : 0.00 %
🔥 WARM (>=5 clics)      : 0.00 %
------------------------------------------------------------
⏱️ Temps total d'évaluation : 1700.68 s


# Évaluation qualitative (cas d'usages concrets)